In [0]:
print("Hospital Readmission project")
print("Databricks notebook is working!")

Hospital Readmission project
Databricks notebook is working!


In [0]:
file_path = "/Volumes/workspace/default/hospital_readmission_data/healthcare_powerbi_master.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

display(df.limit(5))

admission_id,patient_hash,doctor_id,diagnosis_code,admission_date,discharge_date,admission_month,admission_type,length_of_stay,prior_inpatient_admissions,num_medications,num_lab_procedures,num_abnormal_labs,avg_a1c_result,total_charges,insurance_type,icu_stay,readmit_30d,readmission_risk_score,risk_tier,predicted_readmit
ADM_000001,5837878922d7c3dd,DOC_001,E11.9,2025-11-15,2025-11-17,2025-11-01,Elective,2,0,25,10,1,51.05,3005.75,Medicaid,0,0,0.214,LOW_RISK,0
ADM_000002,5afd476a71bc512e,DOC_007,J44.9,2025-10-19,2025-10-22,2025-10-01,Emergency,3,0,15,71,2,66.05,10505.3,Medicaid,1,0,0.407,ELEVATED_RISK,1
ADM_000003,d482a90174a73f5f,DOC_013,I21.9,2025-03-05,2025-03-07,2025-03-01,Urgent,2,4,9,58,1,102.5,2899.32,Private Commercial,0,0,0.394,ELEVATED_RISK,1
ADM_000004,b4a6974c55843f58,DOC_017,M54.5,2025-07-14,2025-07-17,2025-07-01,Trauma Center,3,1,7,16,2,124.1,8940.65,Private Commercial,0,0,0.266,LOW_RISK,0
ADM_000005,167423537bc5c988,DOC_015,J44.9,2025-07-16,2025-07-18,2025-07-01,Urgent,2,2,9,83,1,97.9,11092.17,Medicaid,1,0,0.5,ELEVATED_RISK,1


In [0]:
print("Total records:", df.count())
print("Total columns:", len(df.columns))

df.printSchema()

Total records: 1200
Total columns: 21
root
 |-- admission_id: string (nullable = true)
 |-- patient_hash: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- diagnosis_code: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- admission_month: date (nullable = true)
 |-- admission_type: string (nullable = true)
 |-- length_of_stay: integer (nullable = true)
 |-- prior_inpatient_admissions: integer (nullable = true)
 |-- num_medications: integer (nullable = true)
 |-- num_lab_procedures: integer (nullable = true)
 |-- num_abnormal_labs: integer (nullable = true)
 |-- avg_a1c_result: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- insurance_type: string (nullable = true)
 |-- icu_stay: integer (nullable = true)
 |-- readmit_30d: integer (nullable = true)
 |-- readmission_risk_score: double (nullable = true)
 |-- risk_tier: string (nullable = true)
 |-- predicted_readmit: integer

In [0]:
df.groupBy("readmit_30d") \
  .count() \
  .orderBy("readmit_30d") \
  .show()

+-----------+-----+
|readmit_30d|count|
+-----------+-----+
|          0|  881|
|          1|  319|
+-----------+-----+



In [0]:
from pyspark.sql.functions import avg, round

df.select(
    round(avg("length_of_stay"), 2)
    .alias("average_length_of_stay")
).show()

+----------------------+
|average_length_of_stay|
+----------------------+
|                  4.17|
+----------------------+



In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.default.hospital_readmission")

In [0]:
spark.sql("""
DESCRIBE DETAIL workspace.default.hospital_readmission
""").show(truncate=False)

+------+------------------------------------+--------------------------------------+-----------+--------+-----------------------+-------------------+----------------+-----------------+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                                  |description|location|createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                                                                                                                                                                 |minReaderVersion|minWriterVersion|tableFeatures                            |statistics     

In [0]:
spark.sql("""
SELECT readmit_30d, COUNT(*) AS patient_count
FROM workspace.default.hospital_readmission
GROUP BY readmit_30d
ORDER BY readmit_30d
""").show()

+-----------+-------------+
|readmit_30d|patient_count|
+-----------+-------------+
|          0|          881|
|          1|          319|
+-----------+-------------+



In [0]:
from pyspark.sql import functions as F

table_name = "workspace.default.hospital_readmission"

df_delta = spark.table(table_name)

print("Total records:", df_delta.count())
print("Total columns:", len(df_delta.columns))

df_delta.printSchema()
display(df_delta.limit(5))

Total records: 1200
Total columns: 21
root
 |-- admission_id: string (nullable = true)
 |-- patient_hash: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- diagnosis_code: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- admission_month: date (nullable = true)
 |-- admission_type: string (nullable = true)
 |-- length_of_stay: integer (nullable = true)
 |-- prior_inpatient_admissions: integer (nullable = true)
 |-- num_medications: integer (nullable = true)
 |-- num_lab_procedures: integer (nullable = true)
 |-- num_abnormal_labs: integer (nullable = true)
 |-- avg_a1c_result: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- insurance_type: string (nullable = true)
 |-- icu_stay: integer (nullable = true)
 |-- readmit_30d: integer (nullable = true)
 |-- readmission_risk_score: double (nullable = true)
 |-- risk_tier: string (nullable = true)
 |-- predicted_readmit: integer

admission_id,patient_hash,doctor_id,diagnosis_code,admission_date,discharge_date,admission_month,admission_type,length_of_stay,prior_inpatient_admissions,num_medications,num_lab_procedures,num_abnormal_labs,avg_a1c_result,total_charges,insurance_type,icu_stay,readmit_30d,readmission_risk_score,risk_tier,predicted_readmit
ADM_000001,5837878922d7c3dd,DOC_001,E11.9,2025-11-15,2025-11-17,2025-11-01,Elective,2,0,25,10,1,51.05,3005.75,Medicaid,0,0,0.214,LOW_RISK,0
ADM_000002,5afd476a71bc512e,DOC_007,J44.9,2025-10-19,2025-10-22,2025-10-01,Emergency,3,0,15,71,2,66.05,10505.3,Medicaid,1,0,0.407,ELEVATED_RISK,1
ADM_000003,d482a90174a73f5f,DOC_013,I21.9,2025-03-05,2025-03-07,2025-03-01,Urgent,2,4,9,58,1,102.5,2899.32,Private Commercial,0,0,0.394,ELEVATED_RISK,1
ADM_000004,b4a6974c55843f58,DOC_017,M54.5,2025-07-14,2025-07-17,2025-07-01,Trauma Center,3,1,7,16,2,124.1,8940.65,Private Commercial,0,0,0.266,LOW_RISK,0
ADM_000005,167423537bc5c988,DOC_015,J44.9,2025-07-16,2025-07-18,2025-07-01,Urgent,2,2,9,83,1,97.9,11092.17,Medicaid,1,0,0.5,ELEVATED_RISK,1


In [0]:
from pyspark.sql import functions as F

# 1. Check missing values in important columns
important_columns = [
    "admission_id",
    "patient_hash",
    "admission_date",
    "discharge_date",
    "length_of_stay",
    "readmit_30d"
]

null_checks = df_delta.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c + "_nulls")
    for c in important_columns
])

display(null_checks)

# 2. Check duplicate admission IDs
duplicate_ids = (
    df_delta.groupBy("admission_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate admission IDs:", duplicate_ids.count())
display(duplicate_ids.limit(10))

# 3. Check invalid readmission labels
invalid_labels = df_delta.filter(
    ~F.col("readmit_30d").isin(0, 1)
    | F.col("readmit_30d").isNull()
)

print("Invalid readmission labels:", invalid_labels.count())

# 4. Check negative length of stay
invalid_los = df_delta.filter(
    F.col("length_of_stay") < 0
)

print("Negative length-of-stay records:", invalid_los.count())

admission_id_nulls,patient_hash_nulls,admission_date_nulls,discharge_date_nulls,length_of_stay_nulls,readmit_30d_nulls
0,0,0,0,0,0


Duplicate admission IDs: 0


admission_id,count


Invalid readmission labels: 0
Negative length-of-stay records: 0


In [0]:
%sql
SELECT
    readmit_30d,
    COUNT(*) AS patient_count,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM workspace.default.hospital_readmission
GROUP BY readmit_30d
ORDER BY readmit_30d;

readmit_30d,patient_count,percentage
0,881,73.42
1,319,26.58


In [0]:
from pyspark.sql import functions as F

# Read the existing Delta table
source_table = "workspace.default.hospital_readmission"
silver_table = "workspace.default.hospital_readmission_silver"

df_source = spark.table(source_table)

print("Source rows:", df_source.count())
print("Source columns:", len(df_source.columns))

# Verify required columns before transforming
required_columns = [
    "admission_id",
    "patient_hash",
    "readmit_30d"
]

missing_columns = [
    c for c in required_columns
    if c not in df_source.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

df_silver = df_source

# Standardize date columns if present
for column_name in ["admission_date", "discharge_date"]:
    if column_name in df_silver.columns:
        df_silver = df_silver.withColumn(
            column_name,
            F.to_date(F.col(column_name))
        )

# Keep only records with required identifiers
df_silver = df_silver.filter(
    F.col("admission_id").isNotNull()
    & F.col("patient_hash").isNotNull()
)

# Keep only valid binary target labels
df_silver = df_silver.filter(
    F.col("readmit_30d").isin(0, 1)
    & F.col("readmit_30d").isNotNull()
)

# Remove duplicate admission IDs
df_silver = df_silver.dropDuplicates(["admission_id"])

# Replace missing abnormal-lab counts with zero
# only when this column exists
if "num_abnormal_labs" in df_silver.columns:
    df_silver = df_silver.fillna(
        {"num_abnormal_labs": 0}
    )

# Recalculate admission month from the admission date
if "admission_date" in df_silver.columns:
    df_silver = df_silver.withColumn(
        "admission_month",
        F.trunc(F.col("admission_date"), "month")
    )

print("Silver rows:", df_silver.count())
print("Silver columns:", len(df_silver.columns))

display(df_silver.limit(10))

Source rows: 1200
Source columns: 21
Silver rows: 1200
Silver columns: 21


admission_id,patient_hash,doctor_id,diagnosis_code,admission_date,discharge_date,admission_month,admission_type,length_of_stay,prior_inpatient_admissions,num_medications,num_lab_procedures,num_abnormal_labs,avg_a1c_result,total_charges,insurance_type,icu_stay,readmit_30d,readmission_risk_score,risk_tier,predicted_readmit
ADM_000001,5837878922d7c3dd,DOC_001,E11.9,2025-11-15,2025-11-17,2025-11-01,Elective,2,0,25,10,1,51.05,3005.75,Medicaid,0,0,0.214,LOW_RISK,0
ADM_000002,5afd476a71bc512e,DOC_007,J44.9,2025-10-19,2025-10-22,2025-10-01,Emergency,3,0,15,71,2,66.05,10505.3,Medicaid,1,0,0.407,ELEVATED_RISK,1
ADM_000003,d482a90174a73f5f,DOC_013,I21.9,2025-03-05,2025-03-07,2025-03-01,Urgent,2,4,9,58,1,102.5,2899.32,Private Commercial,0,0,0.394,ELEVATED_RISK,1
ADM_000004,b4a6974c55843f58,DOC_017,M54.5,2025-07-14,2025-07-17,2025-07-01,Trauma Center,3,1,7,16,2,124.1,8940.65,Private Commercial,0,0,0.266,LOW_RISK,0
ADM_000005,167423537bc5c988,DOC_015,J44.9,2025-07-16,2025-07-18,2025-07-01,Urgent,2,2,9,83,1,97.9,11092.17,Medicaid,1,0,0.5,ELEVATED_RISK,1
ADM_000006,3909278e04012cf6,DOC_025,J18.9,2025-06-02,2025-06-06,2025-06-01,Emergency,4,0,26,32,0,73.6,9634.97,Private Commercial,0,0,0.331,ELEVATED_RISK,1
ADM_000007,60a1800feb51516e,DOC_015,I50.9,2025-11-23,2025-11-26,2025-11-01,Emergency,3,0,10,39,1,106.0,6813.28,Self-Pay,0,1,0.335,ELEVATED_RISK,1
ADM_000008,f5796c27b5cbd41d,DOC_021,M54.5,2025-04-20,2025-04-24,2025-04-01,Urgent,4,1,14,52,2,48.95,10442.34,Medicaid,0,0,0.316,ELEVATED_RISK,1
ADM_000009,0ed8915cca9b17e7,DOC_002,S06.0,2025-08-25,2025-08-28,2025-08-01,Emergency,3,1,15,35,1,71.25,9778.56,Medicare,0,0,0.468,ELEVATED_RISK,1
ADM_000010,04cd09445b40974f,DOC_010,I50.9,2025-12-04,2025-12-18,2025-12-01,Urgent,14,0,24,21,1,83.2,25776.94,Self-Pay,0,0,0.354,ELEVATED_RISK,1


In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.default.hospital_readmission_silver"
    )
)

print("Silver Delta table saved successfully.")

Silver Delta table saved successfully.


In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT admission_id) AS unique_admissions,
    SUM(CASE WHEN readmit_30d = 1 THEN 1 ELSE 0 END)
        AS readmitted_records,
    ROUND(AVG(length_of_stay), 2)
        AS average_length_of_stay
FROM workspace.default.hospital_readmission_silver;

total_records,unique_admissions,readmitted_records,average_length_of_stay
1200,1200,319,4.17


In [0]:
from pyspark.sql import functions as F

silver_table = "workspace.default.hospital_readmission_silver"
gold_table = "workspace.default.fact_admissions"

silver_df = spark.table(silver_table)

# Confirm required business columns exist
required_columns = [
    "admission_id",
    "patient_hash",
    "readmit_30d",
    "length_of_stay"
]

missing = [
    c for c in required_columns
    if c not in silver_df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

# Keep admission-level records for analytics
fact_df = silver_df

# Add a month key if the date is available
if "admission_date" in fact_df.columns:
    fact_df = fact_df.withColumn(
        "admission_month",
        F.trunc("admission_date", "month")
    )

# Write the Gold fact table
(
    fact_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print("Gold fact table created:", gold_table)
print("Rows:", spark.table(gold_table).count())

Gold fact table created: workspace.default.fact_admissions
Rows: 1200


In [0]:
%sql
SELECT
    COUNT(*) AS total_admissions,
    COUNT(DISTINCT admission_id) AS unique_admissions,
    SUM(CASE WHEN readmit_30d = 1 THEN 1 ELSE 0 END)
        AS readmitted_admissions,
    ROUND(AVG(length_of_stay), 2) AS avg_length_of_stay,
    ROUND(100.0 * AVG(readmit_30d), 2) AS readmission_rate_pct
FROM workspace.default.fact_admissions;

total_admissions,unique_admissions,readmitted_admissions,avg_length_of_stay,readmission_rate_pct
1200,1200,319,4.17,26.58


In [0]:
%sql
SELECT
    diagnosis_code,
    COUNT(*) AS admission_count,
    ROUND(AVG(length_of_stay), 2) AS avg_length_of_stay,
    ROUND(100.0 * AVG(readmit_30d), 2) AS readmission_rate_pct
FROM workspace.default.fact_admissions
GROUP BY diagnosis_code
ORDER BY admission_count DESC;

diagnosis_code,admission_count,avg_length_of_stay,readmission_rate_pct
J18.9,166,3.89,24.7
S06.0,160,4.17,28.75
E11.9,153,4.44,28.76
M54.5,150,3.86,17.33
I50.9,148,4.33,24.32
I21.9,147,4.24,34.69
J44.9,142,4.34,27.46
K52.9,134,4.16,26.87


In [0]:
fact_df = spark.table("workspace.default.fact_admissions")

print(fact_df.columns)
fact_df.printSchema()

['admission_id', 'patient_hash', 'doctor_id', 'diagnosis_code', 'admission_date', 'discharge_date', 'admission_month', 'admission_type', 'length_of_stay', 'prior_inpatient_admissions', 'num_medications', 'num_lab_procedures', 'num_abnormal_labs', 'avg_a1c_result', 'total_charges', 'insurance_type', 'icu_stay', 'readmit_30d', 'readmission_risk_score', 'risk_tier', 'predicted_readmit']
root
 |-- admission_id: string (nullable = true)
 |-- patient_hash: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- diagnosis_code: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- admission_month: date (nullable = true)
 |-- admission_type: string (nullable = true)
 |-- length_of_stay: integer (nullable = true)
 |-- prior_inpatient_admissions: integer (nullable = true)
 |-- num_medications: integer (nullable = true)
 |-- num_lab_procedures: integer (nullable = true)
 |-- num_abnormal_labs: integer (nullable = true)

In [0]:
from pyspark.sql import functions as F

fact_df = spark.table("workspace.default.fact_admissions")

# Patient dimension
if "patient_hash" in fact_df.columns:
    dim_patient = (
        fact_df
        .filter(F.col("patient_hash").isNotNull())
        .select("patient_hash")
        .distinct()
    )

    dim_patient.write.format("delta").mode("overwrite").saveAsTable(
        "workspace.default.dim_patient"
    )

# Doctor dimension
if "doctor_id" in fact_df.columns:
    dim_doctor = (
        fact_df
        .filter(F.col("doctor_id").isNotNull())
        .select("doctor_id")
        .distinct()
    )

    dim_doctor.write.format("delta").mode("overwrite").saveAsTable(
        "workspace.default.dim_doctor"
    )

# Diagnosis dimension
if "diagnosis_code" in fact_df.columns:
    dim_diagnosis = (
        fact_df
        .filter(F.col("diagnosis_code").isNotNull())
        .select("diagnosis_code")
        .distinct()
    )

    dim_diagnosis.write.format("delta").mode("overwrite").saveAsTable(
        "workspace.default.dim_diagnosis"
    )

print("Available dimension tables created.")

Available dimension tables created.


In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,dim_diagnosis,false
default,dim_doctor,false
default,dim_patient,false
default,fact_admissions,false
default,hospital_readmission,false
default,hospital_readmission_silver,false


In [0]:
%sql 
SELECT COUNT(*) AS patient_count
FROM workspace.default.dim_patient;

patient_count
454


In [0]:
df_gold = spark.table("workspace.default.hospital_readmission")

display(df_gold.limit(10))

admission_id,patient_hash,doctor_id,diagnosis_code,admission_date,discharge_date,admission_month,admission_type,length_of_stay,prior_inpatient_admissions,num_medications,num_lab_procedures,num_abnormal_labs,avg_a1c_result,total_charges,insurance_type,icu_stay,readmit_30d,readmission_risk_score,risk_tier,predicted_readmit
ADM_000001,5837878922d7c3dd,DOC_001,E11.9,2025-11-15,2025-11-17,2025-11-01,Elective,2,0,25,10,1,51.05,3005.75,Medicaid,0,0,0.214,LOW_RISK,0
ADM_000002,5afd476a71bc512e,DOC_007,J44.9,2025-10-19,2025-10-22,2025-10-01,Emergency,3,0,15,71,2,66.05,10505.3,Medicaid,1,0,0.407,ELEVATED_RISK,1
ADM_000003,d482a90174a73f5f,DOC_013,I21.9,2025-03-05,2025-03-07,2025-03-01,Urgent,2,4,9,58,1,102.5,2899.32,Private Commercial,0,0,0.394,ELEVATED_RISK,1
ADM_000004,b4a6974c55843f58,DOC_017,M54.5,2025-07-14,2025-07-17,2025-07-01,Trauma Center,3,1,7,16,2,124.1,8940.65,Private Commercial,0,0,0.266,LOW_RISK,0
ADM_000005,167423537bc5c988,DOC_015,J44.9,2025-07-16,2025-07-18,2025-07-01,Urgent,2,2,9,83,1,97.9,11092.17,Medicaid,1,0,0.5,ELEVATED_RISK,1
ADM_000006,3909278e04012cf6,DOC_025,J18.9,2025-06-02,2025-06-06,2025-06-01,Emergency,4,0,26,32,0,73.6,9634.97,Private Commercial,0,0,0.331,ELEVATED_RISK,1
ADM_000007,60a1800feb51516e,DOC_015,I50.9,2025-11-23,2025-11-26,2025-11-01,Emergency,3,0,10,39,1,106.0,6813.28,Self-Pay,0,1,0.335,ELEVATED_RISK,1
ADM_000008,f5796c27b5cbd41d,DOC_021,M54.5,2025-04-20,2025-04-24,2025-04-01,Urgent,4,1,14,52,2,48.95,10442.34,Medicaid,0,0,0.316,ELEVATED_RISK,1
ADM_000009,0ed8915cca9b17e7,DOC_002,S06.0,2025-08-25,2025-08-28,2025-08-01,Emergency,3,1,15,35,1,71.25,9778.56,Medicare,0,0,0.468,ELEVATED_RISK,1
ADM_000010,04cd09445b40974f,DOC_010,I50.9,2025-12-04,2025-12-18,2025-12-01,Urgent,14,0,24,21,1,83.2,25776.94,Self-Pay,0,0,0.354,ELEVATED_RISK,1


In [0]:
print("Total records:", df_gold.count())
print("Total columns:", len(df_gold.columns))
print("Column names:", df_gold.columns)

Total records: 1200
Total columns: 21
Column names: ['admission_id', 'patient_hash', 'doctor_id', 'diagnosis_code', 'admission_date', 'discharge_date', 'admission_month', 'admission_type', 'length_of_stay', 'prior_inpatient_admissions', 'num_medications', 'num_lab_procedures', 'num_abnormal_labs', 'avg_a1c_result', 'total_charges', 'insurance_type', 'icu_stay', 'readmit_30d', 'readmission_risk_score', 'risk_tier', 'predicted_readmit']


In [0]:
from pyspark.sql.functions import count, round, col

readmission_summary = (
    df_gold
    .groupBy("readmit_30d")
    .agg(count("*").alias("patient_count"))
    .orderBy("readmit_30d")
)

display(readmission_summary)

readmit_30d,patient_count
0,881
1,319


In [0]:
total_patients = df_gold.count()

display(
    readmission_summary
    .withColumn(
        "percentage",
        round(col("patient_count") * 100 / total_patients, 2)
    )
)

readmit_30d,patient_count,percentage
0,881,73.42
1,319,26.58


In [0]:
from pyspark.sql.functions import avg, round

df_gold.select(
    round(avg("length_of_stay"), 2)
    .alias("average_length_of_stay")
).show()

+----------------------+
|average_length_of_stay|
+----------------------+
|                  4.17|
+----------------------+



In [0]:
%sql
SELECT
    readmit_30d,
    COUNT(*) AS patient_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM workspace.default.hospital_readmission
GROUP BY readmit_30d
ORDER BY readmit_30d;

readmit_30d,patient_count,percentage
0,881,73.42
1,319,26.58


In [0]:
df_gold = spark.table("workspace.default.hospital_readmission")

print(df_gold.columns)

['admission_id', 'patient_hash', 'doctor_id', 'diagnosis_code', 'admission_date', 'discharge_date', 'admission_month', 'admission_type', 'length_of_stay', 'prior_inpatient_admissions', 'num_medications', 'num_lab_procedures', 'num_abnormal_labs', 'avg_a1c_result', 'total_charges', 'insurance_type', 'icu_stay', 'readmit_30d', 'readmission_risk_score', 'risk_tier', 'predicted_readmit']


In [0]:
from pyspark.sql import functions as F

diagnosis_summary = (
    df_gold
    .groupBy("diagnosis_code")
    .agg(
        F.count("*").alias("total_admissions"),
        F.round(F.avg("length_of_stay"), 2)
            .alias("avg_length_of_stay"),
        F.round(
            F.avg("readmit_30d") * 100, 2
        ).alias("readmission_rate_pct")
    )
    .orderBy(F.desc("total_admissions"))
)

display(diagnosis_summary)

diagnosis_code,total_admissions,avg_length_of_stay,readmission_rate_pct
J18.9,166,3.89,24.7
S06.0,160,4.17,28.75
E11.9,153,4.44,28.76
M54.5,150,3.86,17.33
I50.9,148,4.33,24.32
I21.9,147,4.24,34.69
J44.9,142,4.34,27.46
K52.9,134,4.16,26.87


In [0]:
monthly_summary = (
    df_gold
    .withColumn(
        "admission_month",
        F.date_trunc("month", F.to_date("admission_date"))
    )
    .groupBy("admission_month")
    .agg(
        F.count("*").alias("total_admissions"),
        F.round(
            F.avg("readmit_30d") * 100, 2
        ).alias("readmission_rate_pct")
    )
    .orderBy("admission_month")
)

display(monthly_summary)

admission_month,total_admissions,readmission_rate_pct
2025-01-01T00:00:00.000Z,100,28.0
2025-02-01T00:00:00.000Z,80,36.25
2025-03-01T00:00:00.000Z,99,16.16
2025-04-01T00:00:00.000Z,103,23.3
2025-05-01T00:00:00.000Z,109,28.44
2025-06-01T00:00:00.000Z,97,19.59
2025-07-01T00:00:00.000Z,122,25.41
2025-08-01T00:00:00.000Z,107,21.5
2025-09-01T00:00:00.000Z,109,37.61
2025-10-01T00:00:00.000Z,94,31.91


In [0]:
(
    diagnosis_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.gold_diagnosis_summary"
    )
)

print("Gold diagnosis summary saved successfully")

Gold diagnosis summary saved successfully


In [0]:
%sql
SELECT *
FROM workspace.default.gold_diagnosis_summary
ORDER BY total_admissions DESC;

diagnosis_code,total_admissions,avg_length_of_stay,readmission_rate_pct
J18.9,166,3.89,24.7
S06.0,160,4.17,28.75
E11.9,153,4.44,28.76
M54.5,150,3.86,17.33
I50.9,148,4.33,24.32
I21.9,147,4.24,34.69
J44.9,142,4.34,27.46
K52.9,134,4.16,26.87


In [0]:
from pyspark.sql import functions as F

df_gold = spark.table("workspace.default.hospital_readmission")

print("Total rows:", df_gold.count())
print("Columns:", df_gold.columns)

display(df_gold.limit(10))

Total rows: 1200
Columns: ['admission_id', 'patient_hash', 'doctor_id', 'diagnosis_code', 'admission_date', 'discharge_date', 'admission_month', 'admission_type', 'length_of_stay', 'prior_inpatient_admissions', 'num_medications', 'num_lab_procedures', 'num_abnormal_labs', 'avg_a1c_result', 'total_charges', 'insurance_type', 'icu_stay', 'readmit_30d', 'readmission_risk_score', 'risk_tier', 'predicted_readmit']


admission_id,patient_hash,doctor_id,diagnosis_code,admission_date,discharge_date,admission_month,admission_type,length_of_stay,prior_inpatient_admissions,num_medications,num_lab_procedures,num_abnormal_labs,avg_a1c_result,total_charges,insurance_type,icu_stay,readmit_30d,readmission_risk_score,risk_tier,predicted_readmit
ADM_000001,5837878922d7c3dd,DOC_001,E11.9,2025-11-15,2025-11-17,2025-11-01,Elective,2,0,25,10,1,51.05,3005.75,Medicaid,0,0,0.214,LOW_RISK,0
ADM_000002,5afd476a71bc512e,DOC_007,J44.9,2025-10-19,2025-10-22,2025-10-01,Emergency,3,0,15,71,2,66.05,10505.3,Medicaid,1,0,0.407,ELEVATED_RISK,1
ADM_000003,d482a90174a73f5f,DOC_013,I21.9,2025-03-05,2025-03-07,2025-03-01,Urgent,2,4,9,58,1,102.5,2899.32,Private Commercial,0,0,0.394,ELEVATED_RISK,1
ADM_000004,b4a6974c55843f58,DOC_017,M54.5,2025-07-14,2025-07-17,2025-07-01,Trauma Center,3,1,7,16,2,124.1,8940.65,Private Commercial,0,0,0.266,LOW_RISK,0
ADM_000005,167423537bc5c988,DOC_015,J44.9,2025-07-16,2025-07-18,2025-07-01,Urgent,2,2,9,83,1,97.9,11092.17,Medicaid,1,0,0.5,ELEVATED_RISK,1
ADM_000006,3909278e04012cf6,DOC_025,J18.9,2025-06-02,2025-06-06,2025-06-01,Emergency,4,0,26,32,0,73.6,9634.97,Private Commercial,0,0,0.331,ELEVATED_RISK,1
ADM_000007,60a1800feb51516e,DOC_015,I50.9,2025-11-23,2025-11-26,2025-11-01,Emergency,3,0,10,39,1,106.0,6813.28,Self-Pay,0,1,0.335,ELEVATED_RISK,1
ADM_000008,f5796c27b5cbd41d,DOC_021,M54.5,2025-04-20,2025-04-24,2025-04-01,Urgent,4,1,14,52,2,48.95,10442.34,Medicaid,0,0,0.316,ELEVATED_RISK,1
ADM_000009,0ed8915cca9b17e7,DOC_002,S06.0,2025-08-25,2025-08-28,2025-08-01,Emergency,3,1,15,35,1,71.25,9778.56,Medicare,0,0,0.468,ELEVATED_RISK,1
ADM_000010,04cd09445b40974f,DOC_010,I50.9,2025-12-04,2025-12-18,2025-12-01,Urgent,14,0,24,21,1,83.2,25776.94,Self-Pay,0,0,0.354,ELEVATED_RISK,1


In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.hospital_readmission")

kpi_summary = df.agg(
    F.count("*").alias("total_admissions"),
    F.countDistinct("patient_hash").alias("unique_patients"),
    F.round(F.avg("readmit_30d") * 100, 2).alias("readmission_rate_pct"),
    F.round(F.avg("length_of_stay"), 2).alias("avg_length_of_stay")
)

display(kpi_summary)

total_admissions,unique_patients,readmission_rate_pct,avg_length_of_stay
1200,454,26.58,4.17


In [0]:
kpi_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_readmission_kpis")

In [0]:
%sql
SELECT * FROM workspace.default.gold_readmission_kpis;

total_admissions,unique_patients,readmission_rate_pct,avg_length_of_stay
1200,454,26.58,4.17


In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.hospital_readmission")

# Create an aggregated dataset for Power BI
powerbi_summary = df.agg(
    F.count("*").alias("total_admissions"),
    F.countDistinct("patient_hash").alias("unique_patients"),
    F.sum("readmit_30d").alias("readmissions_30d"),
    F.round(F.avg("readmit_30d") * 100, 2).alias("readmission_rate_pct"),
    F.round(F.avg("length_of_stay"), 2).alias("avg_length_of_stay")
)

display(powerbi_summary)

total_admissions,unique_patients,readmissions_30d,readmission_rate_pct,avg_length_of_stay
1200,454,319,26.58,4.17


In [0]:
powerbi_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_powerbi_summary")

In [0]:
%sql
SELECT * FROM workspace.default.gold_powerbi_summary;

total_admissions,unique_patients,readmissions_30d,readmission_rate_pct,avg_length_of_stay
1200,454,319,26.58,4.17


In [0]:
output_path = "/Volumes/workspace/default/hospital_readmission_data/gold_powerbi_summary_csv"

spark.table("workspace.default.gold_powerbi_summary") \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("Export completed:", output_path)

Export completed: /Volumes/workspace/default/hospital_readmission_data/gold_powerbi_summary_csv


In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.hospital_readmission")

# Keep useful columns that actually exist in the source
desired_columns = [
    "admission_id",
    "patient_hash",
    "admission_date",
    "discharge_date",
    "diagnosis_code",
    "length_of_stay",
    "readmit_30d",
    "num_abnormal_labs"
]

available_columns = [c for c in desired_columns if c in df.columns]

powerbi_detail = df.select(*available_columns)

print("Columns selected:", available_columns)
print("Total records:", powerbi_detail.count())

display(powerbi_detail.limit(10))

Columns selected: ['admission_id', 'patient_hash', 'admission_date', 'discharge_date', 'diagnosis_code', 'length_of_stay', 'readmit_30d', 'num_abnormal_labs']
Total records: 1200


admission_id,patient_hash,admission_date,discharge_date,diagnosis_code,length_of_stay,readmit_30d,num_abnormal_labs
ADM_000001,5837878922d7c3dd,2025-11-15,2025-11-17,E11.9,2,0,1
ADM_000002,5afd476a71bc512e,2025-10-19,2025-10-22,J44.9,3,0,2
ADM_000003,d482a90174a73f5f,2025-03-05,2025-03-07,I21.9,2,0,1
ADM_000004,b4a6974c55843f58,2025-07-14,2025-07-17,M54.5,3,0,2
ADM_000005,167423537bc5c988,2025-07-16,2025-07-18,J44.9,2,0,1
ADM_000006,3909278e04012cf6,2025-06-02,2025-06-06,J18.9,4,0,0
ADM_000007,60a1800feb51516e,2025-11-23,2025-11-26,I50.9,3,1,1
ADM_000008,f5796c27b5cbd41d,2025-04-20,2025-04-24,M54.5,4,0,2
ADM_000009,0ed8915cca9b17e7,2025-08-25,2025-08-28,S06.0,3,0,1
ADM_000010,04cd09445b40974f,2025-12-04,2025-12-18,I50.9,14,0,1


In [0]:
powerbi_detail.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_powerbi_admissions")

In [0]:
%sql
SELECT * 
FROM workspace.default.gold_powerbi_admissions
LIMIT 20;

admission_id,patient_hash,admission_date,discharge_date,diagnosis_code,length_of_stay,readmit_30d,num_abnormal_labs
ADM_000001,5837878922d7c3dd,2025-11-15,2025-11-17,E11.9,2,0,1
ADM_000002,5afd476a71bc512e,2025-10-19,2025-10-22,J44.9,3,0,2
ADM_000003,d482a90174a73f5f,2025-03-05,2025-03-07,I21.9,2,0,1
ADM_000004,b4a6974c55843f58,2025-07-14,2025-07-17,M54.5,3,0,2
ADM_000005,167423537bc5c988,2025-07-16,2025-07-18,J44.9,2,0,1
ADM_000006,3909278e04012cf6,2025-06-02,2025-06-06,J18.9,4,0,0
ADM_000007,60a1800feb51516e,2025-11-23,2025-11-26,I50.9,3,1,1
ADM_000008,f5796c27b5cbd41d,2025-04-20,2025-04-24,M54.5,4,0,2
ADM_000009,0ed8915cca9b17e7,2025-08-25,2025-08-28,S06.0,3,0,1
ADM_000010,04cd09445b40974f,2025-12-04,2025-12-18,I50.9,14,0,1


In [0]:
output_path = "/Volumes/workspace/default/hospital_readmission_data/gold_powerbi_admissions_csv"

spark.table("workspace.default.gold_powerbi_admissions") \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("Export completed:", output_path)

Export completed: /Volumes/workspace/default/hospital_readmission_data/gold_powerbi_admissions_csv


In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.hospital_readmission")

if "admission_date" not in df.columns:
    raise ValueError("admission_date column is missing")

monthly_trend = (
    df.withColumn(
        "admission_month",
        F.date_trunc("month", F.to_date("admission_date"))
    )
    .groupBy("admission_month")
    .agg(
        F.count("*").alias("total_admissions"),
        F.sum("readmit_30d").alias("readmissions_30d"),
        F.round(
            F.avg("readmit_30d") * 100, 2
        ).alias("readmission_rate_pct"),
        F.round(
            F.avg("length_of_stay"), 2
        ).alias("avg_length_of_stay")
    )
    .orderBy("admission_month")
)

display(monthly_trend)

admission_month,total_admissions,readmissions_30d,readmission_rate_pct,avg_length_of_stay
2025-01-01T00:00:00.000Z,100,28,28.0,4.06
2025-02-01T00:00:00.000Z,80,29,36.25,4.13
2025-03-01T00:00:00.000Z,99,16,16.16,3.96
2025-04-01T00:00:00.000Z,103,24,23.3,4.43
2025-05-01T00:00:00.000Z,109,31,28.44,4.16
2025-06-01T00:00:00.000Z,97,19,19.59,4.14
2025-07-01T00:00:00.000Z,122,31,25.41,4.03
2025-08-01T00:00:00.000Z,107,23,21.5,4.26
2025-09-01T00:00:00.000Z,109,41,37.61,4.43
2025-10-01T00:00:00.000Z,94,30,31.91,4.12


In [0]:
monthly_trend.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_monthly_readmission_trend")

In [0]:
%sql
SELECT *
FROM workspace.default.gold_monthly_readmission_trend
ORDER BY admission_month;

admission_month,total_admissions,readmissions_30d,readmission_rate_pct,avg_length_of_stay
2025-01-01T00:00:00.000Z,100,28,28.0,4.06
2025-02-01T00:00:00.000Z,80,29,36.25,4.13
2025-03-01T00:00:00.000Z,99,16,16.16,3.96
2025-04-01T00:00:00.000Z,103,24,23.3,4.43
2025-05-01T00:00:00.000Z,109,31,28.44,4.16
2025-06-01T00:00:00.000Z,97,19,19.59,4.14
2025-07-01T00:00:00.000Z,122,31,25.41,4.03
2025-08-01T00:00:00.000Z,107,23,21.5,4.26
2025-09-01T00:00:00.000Z,109,41,37.61,4.43
2025-10-01T00:00:00.000Z,94,30,31.91,4.12


In [0]:
output_path = "/Volumes/workspace/default/hospital_readmission_data/gold_monthly_trend_csv"

spark.table("workspace.default.gold_monthly_readmission_trend") \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("Monthly trend export completed:", output_path)

Monthly trend export completed: /Volumes/workspace/default/hospital_readmission_data/gold_monthly_trend_csv


In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.hospital_readmission")

if "diagnosis_code" not in df.columns:
    print("Diagnosis column missing. Available columns:", df.columns)

else:
    diagnosis_summary = (
        df.groupBy("diagnosis_code")
        .agg(
            F.count("*").alias("total_admissions"),
            F.sum("readmit_30d").alias("readmissions_30d"),
            F.round(F.avg("readmit_30d") * 100, 2)
                .alias("readmission_rate_pct"),
            F.round(F.avg("length_of_stay"), 2)
                .alias("avg_length_of_stay")
        )
    )

    diagnosis_summary.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("workspace.default.gold_diagnosis_summary")

    display(
        spark.table("workspace.default.gold_diagnosis_summary")
    )

    print("Diagnosis summary saved successfully.")

diagnosis_code,total_admissions,readmissions_30d,readmission_rate_pct,avg_length_of_stay
J18.9,166,41,24.7,3.89
S06.0,160,46,28.75,4.17
K52.9,134,36,26.87,4.16
E11.9,153,44,28.76,4.44
I21.9,147,51,34.69,4.24
M54.5,150,26,17.33,3.86
I50.9,148,36,24.32,4.33
J44.9,142,39,27.46,4.34


Diagnosis summary saved successfully.


In [0]:
tables = {
    "gold_powerbi_summary":
        "/Volumes/workspace/default/hospital_readmission_data/gold_powerbi_summary_csv",
    "gold_powerbi_admissions":
        "/Volumes/workspace/default/hospital_readmission_data/gold_powerbi_admissions_csv",
    "gold_monthly_readmission_trend":
        "/Volumes/workspace/default/hospital_readmission_data/gold_monthly_trend_csv",
    "gold_diagnosis_summary":
        "/Volumes/workspace/default/hospital_readmission_data/gold_diagnosis_summary_csv"
}

for table, path in tables.items():
    try:
        spark.table(f"workspace.default.{table}") \
            .coalesce(1) \
            .write \
            .mode("overwrite") \
            .option("header", True) \
            .csv(path)

        print(f"Export successful: {table}")

    except Exception as e:
        print(f"Export failed: {table} | {str(e)[:300]}")

Export successful: gold_powerbi_summary
Export successful: gold_powerbi_admissions
Export successful: gold_monthly_readmission_trend
Export successful: gold_diagnosis_summary


In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

+--------+------------------------------+-----------+
|database|tableName                     |isTemporary|
+--------+------------------------------+-----------+
|default |dim_diagnosis                 |false      |
|default |dim_doctor                    |false      |
|default |dim_patient                   |false      |
|default |fact_admissions               |false      |
|default |gold_diagnosis_summary        |false      |
|default |gold_monthly_readmission_trend|false      |
|default |gold_powerbi_admissions       |false      |
|default |gold_powerbi_summary          |false      |
|default |gold_readmission_kpis         |false      |
|default |hospital_readmission          |false      |
|default |hospital_readmission_silver   |false      |
+--------+------------------------------+-----------+



In [0]:
tables = [
    "hospital_readmission",
    "hospital_readmission_silver",
    "fact_admissions",
    "gold_powerbi_summary",
    "gold_powerbi_admissions",
    "gold_monthly_readmission_trend",
    "gold_diagnosis_summary"
]

for table in tables:
    try:
        df = spark.table(f"workspace.default.{table}")
        print(f"{table}: {df.count()} rows")
    except Exception as e:
        print(f"{table}: NOT FOUND - {str(e)[:150]}")

hospital_readmission: 1200 rows
hospital_readmission_silver: 1200 rows
fact_admissions: 1200 rows
gold_powerbi_summary: 1 rows
gold_powerbi_admissions: 1200 rows
gold_monthly_readmission_trend: 12 rows
gold_diagnosis_summary: 8 rows


#ML evaluation files

In [0]:
import os

for root, dirs, files in os.walk("/Workspace"):
    for file in files:
        if file.endswith(".py") and any(
            word in file.lower()
            for word in ["model", "readmission", "train", "evaluat"]
        ):
            print(os.path.join(root, file))

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.hospital_readmission")

print("Total rows:", df.count())
print("Columns:", df.columns)

display(
    df.groupBy("readmit_30d")
      .count()
      .orderBy("readmit_30d")
)

Total rows: 1200
Columns: ['admission_id', 'patient_hash', 'doctor_id', 'diagnosis_code', 'admission_date', 'discharge_date', 'admission_month', 'admission_type', 'length_of_stay', 'prior_inpatient_admissions', 'num_medications', 'num_lab_procedures', 'num_abnormal_labs', 'avg_a1c_result', 'total_charges', 'insurance_type', 'icu_stay', 'readmit_30d', 'readmission_risk_score', 'risk_tier', 'predicted_readmit']


readmit_30d,count
0,881
1,319


In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

# Load the existing Delta table
df = spark.table("workspace.default.hospital_readmission")

# Select features that actually exist in the dataset
categorical_candidates = [
    "diagnosis_group", "admission_type",
    "insurance_type", "age_group", "gender"
]
numeric_candidates = [
    "age", "length_of_stay", "prior_inpatient_admissions",
    "num_medications", "num_lab_procedures",
    "num_abnormal_labs", "avg_a1c_result",
    "total_charges", "icu_stay", "base_risk"
]

categorical_cols = [c for c in categorical_candidates if c in df.columns]
numeric_cols = [c for c in numeric_candidates if c in df.columns]

target = "readmit_30d"

if target not in df.columns:
    raise ValueError(f"Missing target column: {target}")

if not categorical_cols and not numeric_cols:
    raise ValueError("No supported feature columns found.")

# Keep target values 0 and 1, remove null target rows
df = df.filter(
    F.col(target).isNotNull() &
    F.col(target).cast("int").isin(0, 1)
)

df = df.withColumn(target, F.col(target).cast("double"))

# Split data before fitting preprocessing/model
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

stages = []
encoded_cols = []

# Impute numeric nulls
if numeric_cols:
    imputed_cols = [f"{c}_imputed" for c in numeric_cols]
    imputer = Imputer(
        inputCols=numeric_cols,
        outputCols=imputed_cols,
        strategy="median"
    )
    stages.append(imputer)
    encoded_cols.extend(imputed_cols)

# Encode categorical columns
for c in categorical_cols:
    indexed = f"{c}_indexed"
    encoded = f"{c}_encoded"

    stages.append(
        StringIndexer(
            inputCol=c,
            outputCol=indexed,
            handleInvalid="keep"
        )
    )
    stages.append(
        OneHotEncoder(
            inputCols=[indexed],
            outputCols=[encoded],
            handleInvalid="keep"
        )
    )
    encoded_cols.append(encoded)

stages.append(
    VectorAssembler(
        inputCols=encoded_cols,
        outputCol="features",
        handleInvalid="keep"
    )
)

stages.append(
    RandomForestClassifier(
        labelCol=target,
        featuresCol="features",
        predictionCol="prediction",
        probabilityCol="probability",
        numTrees=100,
        maxDepth=5,
        seed=42
    )
)

pipeline = Pipeline(stages=stages)
model = pipeline.fit(train_df)
predictions = model.transform(test_df).cache()

print("Training rows:", train_df.count())
print("Test rows:", test_df.count())
print("Selected categorical features:", categorical_cols)
print("Selected numeric features:", numeric_cols)

display(predictions.select(target, "prediction", "probability").limit(10))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8267588532986633>, line 104
    102 pipeline = Pipeline(stages=stages)
    103 model = pipeline.fit(train_df)
--> 104 predictions = model.transform(test_df).cache()
    106 print("Training rows:", train_df.count())
    107 print("Test rows:", test_df.count())

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2379, in DataFrame.cache(self)
   2378 def cache(self) -> ParentDataFrame:
-> 2379     return self.persist()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2410, in DataFrame.persist(self, storageLevel)
   2408     command = pb2.Command()
   2409     command.create_dataframe_snapshot_command.relation.CopyFrom(relation)
-> 2410     self._session.client.execute_command(command)
   2411     return self
   2412 # END-EDGE

File /databricks/python/

In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, Imputer
)
from pyspark.ml.classification import RandomForestClassifier

# Load existing Databricks Delta table
df = spark.table("workspace.default.hospital_readmission")

target = "readmit_30d"

categorical_candidates = [
    "diagnosis_group", "admission_type",
    "insurance_type", "age_group", "gender"
]

numeric_candidates = [
    "age", "length_of_stay", "prior_inpatient_admissions",
    "num_medications", "num_lab_procedures",
    "num_abnormal_labs", "avg_a1c_result",
    "total_charges", "icu_stay", "base_risk"
]

if target not in df.columns:
    raise ValueError(f"Missing target column: {target}")

cat_cols = [c for c in categorical_candidates if c in df.columns]
num_cols = [c for c in numeric_candidates if c in df.columns]

if not cat_cols and not num_cols:
    raise ValueError(f"No supported features found. Columns: {df.columns}")

df = df.filter(
    F.col(target).isNotNull() &
    F.col(target).cast("int").isin(0, 1)
).withColumn(target, F.col(target).cast("double"))

# Split data into training and test sets
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

stages = []
feature_cols = []

# Fill missing numeric values
if num_cols:
    imputed_cols = [c + "_imputed" for c in num_cols]
    stages.append(
        Imputer(
            inputCols=num_cols,
            outputCols=imputed_cols,
            strategy="median"
        )
    )
    feature_cols.extend(imputed_cols)

# Encode categorical values
for c in cat_cols:
    idx = c + "_idx"
    enc = c + "_enc"

    stages.append(
        StringIndexer(
            inputCol=c,
            outputCol=idx,
            handleInvalid="keep"
        )
    )
    stages.append(
        OneHotEncoder(
            inputCols=[idx],
            outputCols=[enc],
            handleInvalid="keep"
        )
    )
    feature_cols.append(enc)

stages.append(
    VectorAssembler(
        inputCols=feature_cols,
        outputCol="features",
        handleInvalid="keep"
    )
)

stages.append(
    RandomForestClassifier(
        labelCol=target,
        featuresCol="features",
        predictionCol="prediction",
        probabilityCol="probability",
        numTrees=100,
        maxDepth=5,
        seed=42
    )
)

pipeline = Pipeline(stages=stages)
model = pipeline.fit(train_df)

# IMPORTANT: no cache() or persist()
predictions = model.transform(test_df)

print("Training rows:", train_df.count())
print("Test rows:", test_df.count())
print("Categorical features:", cat_cols)
print("Numeric features:", num_cols)

display(predictions.select(target, "prediction", "probability").limit(10))

Training rows: 964
Test rows: 236
Categorical features: ['admission_type', 'insurance_type']
Numeric features: ['length_of_stay', 'prior_inpatient_admissions', 'num_medications', 'num_lab_procedures', 'num_abnormal_labs', 'avg_a1c_result', 'total_charges', 'icu_stay']


readmit_30d,prediction,probability
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7221312145592632"",""0.27786878544073673""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7469002126590257"",""0.25309978734097427""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8056584155258121"",""0.19434158447418795""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6778069722007446"",""0.3221930277992554""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.817669891714192"",""0.18233010828580792""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8063145607669666"",""0.19368543923303336""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.792729386713172"",""0.20727061328682816""]}"
1.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.6929529924939888"",""0.30704700750601127""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.5848405117707111"",""0.4151594882292889""]}"
1.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7737368854788865"",""0.22626311452111345""]}"


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.evaluation import BinaryClassificationEvaluator

tp = predictions.filter(
    (F.col(target) == 1) & (F.col("prediction") == 1)
).count()

fp = predictions.filter(
    (F.col(target) == 0) & (F.col("prediction") == 1)
).count()

fn = predictions.filter(
    (F.col(target) == 1) & (F.col("prediction") == 0)
).count()

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) else 0.0
)

auc = BinaryClassificationEvaluator(
    labelCol=target,
    rawPredictionCol="probability",
    metricName="areaUnderROC"
).evaluate(predictions)

print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1-score:", round(f1, 3))
print("ROC-AUC:", round(auc, 3))
print("Recall target >= 0.70:", "MET" if recall >= 0.70 else "NOT MET")

Precision: 0.333
Recall: 0.017
F1-score: 0.033
ROC-AUC: 0.565
Recall target >= 0.70: NOT MET
